In [18]:
import requests
import pickle as pkl

import pandas as pd
import pickle as pkl
import os
import json
from natsort import natsorted
import re
from tqdm.auto import tqdm
from ast import literal_eval
from collections import defaultdict


In [19]:
import spacy

# Load the SpaCy language model (replace 'en_core_web_sm' with a larger model if needed)
nlp = spacy.load('en_core_web_md')


In [20]:
with open('hackerNews_updated.json', 'r') as f:
    file=json.load(f)

In [21]:
def remove_between_tags(text):
    # Remove everything between < and > including the symbols
    text = re.sub(r'<.*?>', '', text)
    # Remove any extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [22]:
for items in tqdm(file.keys()):
    content_list=file[items]['content']
    temp=[]
    for sentences in content_list:
        fixed=remove_between_tags(sentences)
        if fixed is not None:
            temp.append(remove_between_tags(sentences))
    file[items]['fixed_content']=temp

100%|██████████| 13598/13598 [00:02<00:00, 6728.20it/s] 


In [23]:
results=os.listdir('outputs')

In [24]:
results=natsorted(results)

In [25]:
pkls_list=[]
for name in results:
    with open('outputs/{}'.format(name), 'rb') as f:
        pkls_list.append(pkl.load(f))

In [26]:
pased_data_list=[]
for k,items in enumerate(pkls_list):
    try:
        raw_json_str=items[0]
        cleaned_json_str = raw_json_str.strip('```json\n').strip('\n```')
        parsed_data = json.loads(cleaned_json_str)
        pased_data_list.append(parsed_data)
    except:
        pased_data_list.append({})
        print(k)
        print(cleaned_json_str)

1463
{
    "Named_Entities": {
        "software": [
            "Orcus",
            "Remote Access Trojan",
            "RAT",
            "Orcus RAT",
            "Trojans",
            "worms",
            "viruses",
            "Or
2115
{
    "Named_Entities": {
        "software": [
            "Photo Station"
        ],
        "hardware": [
            "NAS326",
            "NAS540",
            "NAS542",
            "GS1200 series switches"
        ],
        "software_vulnerabilities": [
            "CVE-2022-34747",
            "CVE-2022-30526",
            "CVE-2022-2030",
            "CVE-2022-0823"
        ],
        "hardware_vulnerabilities": []
    },
    "Relations": [
        "NAS326, CVE-2022-34747",
        "NAS540, CVE-2022-34747",
        "NAS542, CVE-2022-34747",
        "GS1200 series switches, CVE-2022-


In [27]:
software=[]
hardware=[]
software_vulnerabilities=[]
hardware_vulnerabilities=[]
relations=[]
for items in pased_data_list:
    try:
        software.extend(items['Named_Entities']['software'])
        hardware.extend(items['Named_Entities']['hardware'])
        software_vulnerabilities.extend(items['Named_Entities']['software_vulnerabilities'])
        hardware_vulnerabilities.extend(items['Named_Entities']['hardware_vulnerabilities'])
        relations.extend(items['Relations'])
    except:
        pass
        # try:
        #     print(items)
        #     software.extend(items['Named_Entities']['software'])
        #     hardware.extend(items['Named_Entities']['hardware'])
        #     software_vulnerabilities.extend(items['Named_Entities']['software_security_vulnerabilities'])
        #     hardware_vulnerabilities.extend(items['Named_Entities']['hardware_security_vulnerabilities'])
        #     relations.extend(items['Relations'])
        # except:
        #     software.append([])
        #     hardware.append([])
        #     software_vulnerabilities.append([])
        #     hardware_vulnerabilities.append([])
        #     relations.append([])
            

In [28]:
print(len(software))
print(len(hardware))
print(len(software_vulnerabilities))
print(len(hardware_vulnerabilities))
print(len(relations))


38685
2127
5669
279
13293


In [29]:
software=list(set(software))
hardware=list(set(hardware))
software_vulnerabilities=list(set(software_vulnerabilities))
hardware_vulnerabilities=list(set(hardware_vulnerabilities))
relations=list(set(relations))

In [30]:
print(len(software))
print(len(hardware))
print(len(software_vulnerabilities))
print(len(hardware_vulnerabilities))
print(len(relations))


12364
1474
3533
248
11012


In [31]:
hardware_vulnerabilities

['CVE-2022-30525',
 'ASUS Driver7.sys',
 'cache contention',
 'CVE-2023-33009',
 'implants',
 'SMASH',
 'CVE-2023-45208',
 'CVE-2015-2291',
 'multiple exploits',
 'vulnerable IoT devices',
 'power management tampering techniques',
 'side-channel attacks',
 'PACMAN',
 'flaws in network-attached storage (NAS) devices',
 'weak hardware security',
 'authentication bypass vulnerability',
 'hacking automotive systems',
 'CVE-2022-29854',
 'CVE-2023-27989',
 'CVE-2023-20079',
 'bootkits',
 'CVE-2021-3011',
 'multiple high-impact security flaws',
 'CVE-2021-45382',
 'CVE-2022-29855',
 'CVE-2021-1975',
 'TEMPEST attack',
 'CVE-2021-4045',
 'Bluetooth signals fingerprinting',
 'CVE-2021-40112',
 'BIOS control register',
 'security vulnerability in the Firebox firmware',
 'CVE-2022-20210',
 'flaw in the firmware upgrade mechanism',
 'CVE-2017-17215',
 'CVE-2022-46649',
 'CVE-2021-34795',
 'CVE-2018-19322',
 'critical security vulnerabilities',
 'persistent firmware implants',
 'CVE-2021-1924',
 '

In [ ]:
import requests
import time
from tqdm import tqdm
from collections import defaultdict

def chunk_list(lst, chunk_size):
    """Splits a list into smaller chunks of size `chunk_size`."""
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

def get_wikipedia_introductions(entities, disambiguation_context=""):
    """
    Fetches the introduction text of multiple Wikipedia entities.
    If an entity is a disambiguation page, it will re-search the entity with the provided context.

    :param entities: List of Wikipedia page titles
    :param disambiguation_context: Additional text to refine search on disambiguation pages
    :return: Dictionary mapping entity names to a list of introduction texts.
    """
    base_url = "https://en.wikipedia.org/w/api.php"
    introductions = defaultdict(list)  # Store multiple intros per entity

    for batch in tqdm(list(chunk_list(entities, 50)), desc="Processing batches"):
        titles = "|".join(batch)

        params = {
            "format": "json",
            "action": "query",
            "prop": "extracts|pageprops",
            "exintro": True,
            "explaintext": True,
            "redirects": 1,
            "titles": titles
        }

        try:
            response = requests.get(base_url, params=params)

            # Handle rate limiting (HTTP 429)
            if response.status_code == 429:
                print("[ERROR] Rate limit exceeded. Retrying in 10 seconds...")
                time.sleep(10)
                return get_wikipedia_introductions(entities, disambiguation_context)  # Retry request

            elif response.status_code != 200:
                print(f"[ERROR] API returned status code {response.status_code}: {response.text}")
                continue

            data = response.json()

            # Check for explicit API error messages
            if "error" in data:
                print(f"[ERROR] Wikipedia API Error: {data['error']['info']}")
                continue

            # Extract page details
            pages = data.get("query", {}).get("pages", {})

            for page_id, page_info in pages.items():
                title = page_info.get("title", "Unknown")
                extract = page_info.get("extract", "").strip()
                is_disambiguation = "disambiguation" in page_info.get("pageprops", {})

                if is_disambiguation:
                    # If a disambiguation context is provided, retry with refined search
                    if disambiguation_context:
                        refined_search = f"{title} {disambiguation_context}"
                        print(f"[INFO] {title} is a disambiguation page. Retrying search with '{refined_search}'...")
                        refined_result = get_wikipedia_introductions([refined_search], disambiguation_context)
                        #print(refined_result)
                        introductions[title] = refined_result.get(refined_search, ["(No refined match found.)"])
                    else:
                        print(f"[INFO] {title} is a disambiguation page but no search context was provided.")
                        introductions[title] = ["(Disambiguation detected, but no context to refine search.)"]

                elif extract:
                    introductions[title].append(extract)
                else:
                    introductions[title].append("No introduction found.")

        except requests.exceptions.RequestException as e:
            print(f"[ERROR] Network error: {e}")
            continue

    return introductions

# Example usage
# entities = ["Smash", "Implant", "Python (programming language)", "Machine learning"]
disambiguation_context = "software"  # Custom search term for disambiguation pages

introductions = get_wikipedia_introductions(software, disambiguation_context)

# Store in dictionary
# wiki_data = dict(introductions)

# Example Output
# for k,v in introductions.items():
#     if v[0]!='No introduction found.':
#         print(k)
#         print(v)

with open('wikipedia_software.pkl', 'wb') as f:
    pkl.dump(introductions,f)

Processing batches:   0%|          | 0/248 [00:00<?, ?it/s]

[INFO] Jabber is a disambiguation page. Retrying search with 'Jabber software'...


Processing batches: 100%|██████████| 1/1 [00:00<00:00,  8.32it/s]


[INFO] Pax is a disambiguation page. Retrying search with 'Pax software'...


Processing batches:   0%|          | 1/248 [00:00<01:43,  2.39it/s]

...

[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...


[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...


[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. Retrying search with 'Alien software'...
[INFO] Alien is a disambiguation page. R




















Processing batches:  32%|███▏      | 79/248 [06:15<13:23,  4.75s/it]


KeyboardInterrupt: 

In [61]:
import requests
import time
import os
import json
from tqdm import tqdm
from collections import defaultdict

SAVE_FOLDER = "wiki_results"  # Folder for saving progress
os.makedirs(SAVE_FOLDER, exist_ok=True)  # Ensure folder exists

def save_progress(data, filename):
    """Saves the dictionary to a JSON file."""
    with open(os.path.join(SAVE_FOLDER, filename), "w") as f:
        json.dump(data, f, indent=4)

def load_progress(filename):
    """Loads progress from a JSON file, if available."""
    filepath = os.path.join(SAVE_FOLDER, filename)
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            return json.load(f)
    return {}

def chunk_list(lst, chunk_size):
    """Splits a list into smaller chunks of size `chunk_size`."""
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

def get_wikipedia_introductions(entities, disambiguation_context="", batch_size=50, max_retries=5):
    """
    Fetches introductions for multiple Wikipedia entities.
    Handles rate limits, saves progress after each batch, and retries disambiguation pages up to 5 times.

    :param entities: List of Wikipedia page titles
    :param disambiguation_context: Search term for refining disambiguation pages
    :param batch_size: Number of entities per API request (max 50)
    :param max_retries: Maximum retries for handling disambiguation pages
    :return: Dictionary mapping entity names to a list of introduction texts.
    """
    base_url = "https://en.wikipedia.org/w/api.php"
    introductions = defaultdict(list)

    # Load previous progress
    saved_data = load_progress("wiki_data.json")

    for batch in tqdm(list(chunk_list(entities, batch_size)), desc="Processing batches"):
        batch_titles = [title for title in batch if title not in saved_data]  # Skip already processed

        if not batch_titles:
            continue  # Skip batch if all entities are processed
        if 'Alien' in batch_titles:
            batch_titles.remove('Alien')

        titles = "|".join(batch_titles)

        params = {
            "format": "json",
            "action": "query",
            "prop": "extracts|pageprops",
            "exintro": True,
            "explaintext": True,
            "redirects": 1,
            "titles": titles
        }

        retry = True
        while retry:
            try:
                response = requests.get(base_url, params=params)

                # Handle rate limiting (HTTP 429)
                if response.status_code == 429:
                    print("[ERROR] Rate limit exceeded. Waiting for 10 seconds...")
                    time.sleep(10)
                    continue  # Retry the request

                elif response.status_code != 200:
                    print(f"[ERROR] API returned status code {response.status_code}: {response.text}")
                    retry = False
                    continue

                data = response.json()
                retry = False  # Exit retry loop if request is successful

            except requests.exceptions.RequestException as e:
                print(f"[ERROR] Network error: {e}. Retrying in 10 seconds...")
                time.sleep(10)

        # Check for explicit API errors
        if "error" in data:
            print(f"[ERROR] Wikipedia API Error: {data['error']['info']}")
            continue

        # Extract page details
        pages = data.get("query", {}).get("pages", {})

        for page_id, page_info in pages.items():
            title = page_info.get("title", "Unknown")
            extract = page_info.get("extract", "").strip()
            is_disambiguation = "disambiguation" in page_info.get("pageprops", {})

            if is_disambiguation:
                # Attempt to refine search with disambiguation context, with a max of 5 retries
                retries = 0
                refined_result = {}
                while retries < max_retries:
                    if disambiguation_context:
                        refined_search = f"{title} {disambiguation_context}"
                        print(f"[INFO] {title} is a disambiguation page. Retrying search with '{refined_search}' (Attempt {retries + 1}/{max_retries})...")
                        refined_result = get_wikipedia_introductions([refined_search], disambiguation_context)
                        if refined_result.get(refined_search):
                            introductions[title] = refined_result[refined_search]
                            break  # Stop retrying if a valid result is found
                    retries += 1
                    time.sleep(2)  # Small delay between retries

                if retries == max_retries:
                    print(f"[WARNING] Max retries reached for {title}. Storing fallback message.")
                    introductions[title] = ["(Max retries reached. Could not resolve disambiguation.)"]

            elif extract:
                introductions[title].append(extract)
            else:
                introductions[title].append("No introduction found.")

        # Merge progress into saved data
        saved_data.update(introductions)
        save_progress(saved_data, "wiki_data.json")  # Save progress

    return saved_data

# # Example usage
# entities = ["Smash", "Implant", "Python (programming language)", "Machine learning", "AI model"]
disambiguation_context = "software"  # Custom search term for disambiguation pages

introductions = get_wikipedia_introductions(software, disambiguation_context)

with open('wikipedia_software.pkl', 'wb') as f:
    pkl.dump(introductions,f)




Processing batches:   0%|          | 0/248 [00:00<?, ?it/s]

[INFO] Pax is a disambiguation page. Retrying search with 'Pax software' (Attempt 1/5)...


Processing batches:   0%|          | 1/248 [00:00<01:14,  3.31it/s]

[INFO] SH is a disambiguation page. Retrying search with 'SH software' (Attempt 1/5)...


Processing batches: 100%|██████████| 1/1 [00:00<00:00,  7.65it/s]


...

Processing batches: 100%|██████████| 1/1 [00:00<00:00,  7.02it/s]


[INFO] Zzzz is a disambiguation page. Retrying search with 'Zzzz software' (Attempt 1/5)...


Processing batches: 100%|█████████▉| 247/248 [06:05<00:03,  3.76s/it]

[INFO] Mongo is a disambiguation page. Retrying search with 'Mongo software' (Attempt 1/5)...


Processing batches: 100%|██████████| 248/248 [06:06<00:00,  1.48s/it]
